In [3]:
!pip install xgboost



   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.5/72.0 MB 2.5 MB/s eta 0:00:29
    --------------------------------------- 1.0/72.0 MB 2.8 MB/s eta 0:00:26
   - -------------------------------------- 2.4/72.0 MB 4.0 MB/s eta 0:00:18
   -- ------------------------------------- 3.7/72.0 MB 4.7 MB/s eta 0:00:15
   -- ------------------------------------- 5.2/72.0 MB 5.2 MB/s eta 0:00:13
   --- ------------------------------------ 6.8/72.0 MB 5.7 MB/s eta 0:00:12
   --- ------------------------------------ 6.8/72.0 MB 5.7 MB/s eta 0:00:12
   ---- ----------------------------------- 8.1/72.0 MB 5.2 MB/s eta 0:00:13
   ---- ----------------------------------- 8.9/72.0 MB 4.8 MB/s eta 0:00:14
   ---- ----------------------------------- 8.9/72.0 MB 4.8 MB/s eta 0:00:14
   ----- ---------------------------------- 10.0/72.0 MB 4.6 MB/s eta 0:00:14
   ------ --


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install scikit-learn


  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   ------ --------------------------------- 1.3/8.0 MB 3.3 MB/s eta 0:00:03
   ----------- ---------------------------- 2.4/8.0 MB 4.0 MB/s eta 0:00:02
   --------------- ------------------------ 3.1/8.0 MB 4.7 MB/s eta 0:00:02
   ------------------ --------------------- 3.7/8.0 MB 3.7 MB/s eta 0:00:02
   -------------------------- ------------- 5.2/8.0 MB 4.2 MB/s eta 0:00:01
   ---------------------------------- ----- 6.8/8.0 MB 4.8 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 5.0 MB/s  0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -----


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ===============================
# Daily Food & Nutrition Predictor
# XGBoost + Tkinter GUI
# Logo + Text Header (Jupyter-safe)
# ===============================

import pandas as pd
import tkinter as tk
from tkinter import ttk, messagebox
from PIL import Image, ImageTk

from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor


# -------------------------------
# Step 1: Load Dataset
# -------------------------------
DATA_PATH = "D:/STUDY/UOL2025/CV_Course/nutrition_project/daily_food_nutrition_dataset.csv"
df = pd.read_csv(DATA_PATH, on_bad_lines='skip')
df.fillna(0, inplace=True)


# -------------------------------
# Step 2: Encode Categorical Columns
# -------------------------------
le_food = LabelEncoder()
le_category = LabelEncoder()

df['Food_Item_encoded'] = le_food.fit_transform(df['Food_Item'])
df['Category_encoded'] = le_category.fit_transform(df['Category'].astype(str))


# -------------------------------
# Step 3: Features & Targets
# -------------------------------
X = df[['Food_Item_encoded', 'Category_encoded']]
y = df[
    [
        'Calories (kcal)', 'Protein (g)', 'Carbohydrates (g)', 'Fat (g)',
        'Fiber (g)', 'Sugars (g)', 'Sodium (mg)',
        'Cholesterol (mg)', 'Water_Intake (ml)'
    ]
]


# -------------------------------
# Step 4: Train-Test Split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# -------------------------------
# Step 5: Train XGBoost Model
# -------------------------------
xgb = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=200,
    random_state=42
)

model = MultiOutputRegressor(xgb)
model.fit(X_train, y_train)


# -------------------------------
# Step 6: Prediction Function
# -------------------------------
def predict_nutrition(food_item, category):
    try:
        food_encoded = le_food.transform([food_item])[0]
        category_encoded = le_category.transform([category])[0]
    except ValueError:
        messagebox.showerror("Error", "Food item or category not found!")
        return None

    prediction = model.predict([[food_encoded, category_encoded]])[0]

    return f"""
Predicted Nutrition Information
--------------------------------
Food Item : {food_item}
Category  : {category}

Calories       : {prediction[0]:.2f} kcal
Protein        : {prediction[1]:.2f} g
Carbohydrates  : {prediction[2]:.2f} g
Fat            : {prediction[3]:.2f} g
Fiber          : {prediction[4]:.2f} g
Sugars         : {prediction[5]:.2f} g
Sodium         : {prediction[6]:.2f} mg
Cholesterol    : {prediction[7]:.2f} mg
Water Intake   : {prediction[8]:.2f} ml
"""


# -------------------------------
# Step 7: GUI Logic
# -------------------------------
def on_predict():
    if not food_var.get() or not category_var.get():
        messagebox.showwarning("Input Error", "Please select food and category")
        return

    result = predict_nutrition(food_var.get(), category_var.get())
    output_text.config(state="normal")
    output_text.delete(1.0, tk.END)
    output_text.insert(tk.END, result)
    output_text.config(state="disabled")


# -------------------------------
# Create Main Window
# -------------------------------
root = tk.Tk()
root.title("Daily Food Nutrition Predictor")
root.geometry("650x600")
root.resizable(False, False)


# -------------------------------
# Header: Logo + Text (SAFE)
# -------------------------------
LOGO_PATH = "D:/STUDY/UOL2025/CV_Course/nutrition_project/images.PNG"

logo_img = Image.open(LOGO_PATH).convert("RGBA")
logo_img = logo_img.resize((50, 50))

root.logo_photo = ImageTk.PhotoImage(logo_img)   # 🔥 keep reference

header_frame = tk.Frame(root, bg="white")
header_frame.pack(pady=10)

tk.Label(
    header_frame,
    image=root.logo_photo,
    bg="white"
).pack(side="left", padx=10)

tk.Label(
    header_frame,
    text="Daily Food Nutrition Predictor",
    font=("Arial", 16, "bold"),
    bg="white"
).pack(side="left")


# -------------------------------
# Main Content Frame
# -------------------------------
frame = tk.Frame(root, bg="white", bd=2)
frame.pack(pady=10)


# -------------------------------
# Widgets
# -------------------------------
tk.Label(frame, text="Select Food Item:", bg="white").pack(pady=5)

food_var = tk.StringVar()
ttk.Combobox(
    frame, textvariable=food_var, width=30, state="readonly",
    values=sorted(df['Food_Item'].unique().tolist())
).pack(pady=5)

tk.Label(frame, text="Select Category:", bg="white").pack(pady=5)

category_var = tk.StringVar()
ttk.Combobox(
    frame, textvariable=category_var, width=30, state="readonly",
    values=sorted(df['Category'].unique().tolist())
).pack(pady=5)

tk.Button(
    frame,
    text="Predict Nutrition",
    command=on_predict,
    bg="#4CAF50", fg="white",
    font=("Arial", 10, "bold")
).pack(pady=15)

output_text = tk.Text(frame, height=14, width=60, state="disabled")
output_text.pack(pady=10)


# -------------------------------
# Run Application
# -------------------------------
root.mainloop()
